In [3]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

%load_ext autoreload
%autoreload 2

In [4]:
# Создаем простую среду (один env)
from agent_system.environments import make_envs
from omegaconf import OmegaConf, open_dict
import ray

[2025-11-27 21:21:15] INFO xla_bridge.py:889: Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
[2025-11-27 21:21:15] INFO xla_bridge.py:889: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
2025-11-27 21:21:17.477839: W external/xla/xla/service/gpu/nvptx_compiler.cc:765] The NVIDIA driver's CUDA version is 12.5 which is older than the ptxas CUDA version (12.9.86). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


Loading textures from cache.
Textures successfully loaded from cache.


In [ ]:
try:
    multiprocessing.set_start_method('spawn', force=True)
except RuntimeError:
    # Метод уже установлен, это нормально
    pass

In [ ]:
# Создаем минимальный конфиг для среды БЕЗ оракла (базовая версия)
simple_cfg = OmegaConf.create({
    "env": {
        "env_name": "craftext/CraftextEnv",
        "craftext_settings": "achievements_collect_wood",
        "seed": 42,
        "rollout": {"n": 1},
        "max_steps": 100,
        "resources_per_worker": {  # Ресурсы для каждого env worker (обязательный параметр для make_envs)
            "num_cpus": 0.1,
            "num_gpus": 0,
        },
        "history_length": 1,
    },
    "data": {
        "val_batch_size": 1,
        "train_batch_size": 1,
    },
})

_, val_envs = make_envs(simple_cfg)

In [9]:
# ============================================================================
# ПРИМЕР РАБОТЫ С СРЕДОЙ С ОРАКЛОМ
# ============================================================================

# Создаем конфиг для среды С ораклом
oracle_cfg = OmegaConf.create({
    "env": {
        "env_name": "craftext/CraftextOracleEnv",  # <-- Изменено на Oracle версию
        "craftext_settings": "achievements_collect_wood",
        "seed": 42,
        "rollout": {"n": 1},
        "max_steps": 100,
        "resources_per_worker": {
            "num_cpus": 0.1,
            "num_gpus": 0,
        },
        "history_length": 0,
        "oracle_model_name": "Qwen/Qwen2.5-1.5B-Instruct",  # <-- Модель оракла (опционально)
    },
    "data": {
        "val_batch_size": 1,
        "train_batch_size": 1,
    },
})

print("Создаем среду с ораклом...")
_, oracle_val_envs = make_envs(oracle_cfg)

Создаем среду с ораклом...
[make_envs] Detected Oracle environment: craftext/CraftextOracleEnv
[CraftextMultiProcessEnvOracle] Creating oracle with model Qwen/Qwen2.5-1.5B-Instruct...
(pid=4032182) Loading textures from cache.
(CraftextOracle pid=4032173) [Oracle] Loading model Qwen/Qwen2.5-1.5B-Instruct on cuda...


(raylet) [2025-11-27 21:44:01,149 E 4032094 4032123] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2025-11-27_21-21-28_326942_4029626 is over 95% full, available space: 45.7325 GB; capacity: 1511.58 GB. Object creation will fail if spilling is required.


(pid=4032182) Textures successfully loaded from cache.


(CraftextOracle pid=4032173) Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


(CraftextOracle pid=4032173) [Oracle] Model loaded successfully on cuda


...

(CraftextOracle pid=4032180) Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


(CraftextOracle pid=4032180) [Oracle] Model loaded successfully on cuda


(CraftextWorkerOracle pid=4032177) Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
(CraftextWorkerOracle pid=4032177) Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
(CraftextWorkerOracle pid=4032177) Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
(CraftextWorkerOracle pid=4032177) See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
1it [00:00, 22550.02it/s]e pid=4032177) 
(CraftextWorkerOracle pid=4032177) 2025-11-27 21:44:16,945 - INFO - Initial number of instructions: 1
100%|██████████| 1/1 [00:00<00:00, 13.51it/s]
(CraftextWorkerOracle pid=4032177) 2025-11-27 21:44:17,040 - INFO - Final number of instructions: 1


(CraftextWorkerOracle pid=4032177) _NamespacePath(['/home/n.sorokin/verl-agent/agent_system/environments/env_package/craftext/CrafText-super_igor_env_build/craftext/dataset'])
(CraftextWorkerOracle pid=4032177) /home/n.sorokin/verl-agent/agent_system/environments/env_package/craftext/CrafText-super_igor_env_build/craftext/dataset
(CraftextWorkerOracle pid=4032177) Initial number of instructions: 1
(CraftextWorkerOracle pid=4032177) Final number of instructions: 1
CraftextOracle Evaluating with 1 scenarios.
(CraftextWorkerOracle pid=4032177) Initialized Instruction Wrapper with environment key: 0
(CraftextWorkerOracle pid=4032177) Def instr shape: (768,)


(raylet) [2025-11-27 21:44:21,159 E 4032094 4032123] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2025-11-27_21-21-28_326942_4029626 is over 95% full, available space: 45.748 GB; capacity: 1511.58 GB. Object creation will fail if spilling is required.
(raylet) [2025-11-27 21:44:31,163 E 4032094 4032123] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2025-11-27_21-21-28_326942_4029626 is over 95% full, available space: 45.7479 GB; capacity: 1511.58 GB. Object creation will fail if spilling is required.


In [ ]:
# Пример reset для среды с ораклом
print("=== Reset среды с ораклом ===")
oracle_obs, oracle_info = oracle_val_envs.reset({})

print(f"\nObservation keys: {oracle_obs.keys()}")
print(f"Info keys: {oracle_info[0].keys()}")
print(f"\nInstruction: {oracle_info[0].get('instruction', 'N/A')}")
print(f"\nText observation (первые 500 символов):")
print(oracle_obs['text'][0])
print("\n...")


In [ ]:
# Пример step с вопросом к оракулу
print("=== Step с вопросом к оракулу ===")

# Формируем действие с вопросом (формат, который ожидает LLM)
text_action_with_question = """<question>How do I collect wood in this game?</question>
<action>do</action>"""

# Выполняем step
next_obs, rewards, dones, infos = oracle_val_envs.step([text_action_with_question])

print(f"\nReward: {rewards[0]}")
print(f"Done: {dones[0]}")
print(f"\nInfo keys: {infos[0].keys()}")

# Проверяем, есть ли ответ оракла
if 'oracle_answer' in infos[0]:
    print(f"\n=== Ответ оракла ===")
    print(infos[0]['oracle_answer'])
else:
    print("\nОтвет оракла не найден в info")

# Проверяем, был ли задан вопрос
if 'question' in infos[0]:
    print(f"\n=== Заданный вопрос ===")
    print(infos[0]['question'])

# Показываем новое наблюдение (включает ответ оракла, если был)
print(f"\n=== Новое наблюдение (первые 800 символов) ===")
print(next_obs['text'][0])
print("\n...")


In [1]:
from simple_agent import SimpleAgent

/home/n.sorokin/miniconda3/envs/verl-agent/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 11-27 21:20:33 [__init__.py:239] Automatically detected platform cuda.


2025-11-27 21:20:35,464	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
# Пример использования SimpleAgent

# Создаем Agent
checkpoint_path = ""

# Проверяем наличие чекпоинта и LoRA адаптера
import os
print(f"Проверка чекпоинта: {checkpoint_path}")
print(f"  Чекпоинт существует: {os.path.exists(checkpoint_path)}")

lora_path = os.path.join(checkpoint_path, "actor", "lora_adapter")
print(f"  LoRA путь: {lora_path}")
print(f"  LoRA существует: {os.path.exists(lora_path)}")

if os.path.exists(lora_path):
    adapter_files = os.listdir(lora_path)
    print(f"  LoRA файлы: {adapter_files}")

agent = SimpleAgent(
    checkpoint_path=checkpoint_path,
    base_model_path="Qwen/Qwen2.5-1.5B-Instruct",
    lora_rank=64,
    lora_alpha=64,
    target_modules="all-linear",
    max_response_length=512,
    do_sample=True,  # ИСПРАВЛЕНО: должно совпадать с обучением (val_kwargs.do_sample=True)
    temperature=0.4,  # ИСПРАВЛЕНО: должно совпадать с обучением (val_kwargs.temperature=0.4)
)

# Проверяем, что LoRA загружен
print(f"\nLoRA загружен: {agent.use_lora}")
print(f"LoRA имя: {agent.lora_name}")

Проверка чекпоинта: 
  Чекпоинт существует: False
  LoRA путь: actor/lora_adapter
  LoRA существует: False
Предупреждение: LoRA адаптер не найден в 
Используется базовая модель без LoRA
Инициализация vLLM engine...
INFO 11-27 21:20:46 [config.py:717] This model supports multiple tasks: {'score', 'generate', 'classify', 'embed', 'reward'}. Defaulting to 'generate'.
INFO 11-27 21:20:46 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=16384.
WARNING 11-27 21:20:46 [cuda.py:93] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 11-27 21:20:48 [core.py:58] Initializing a V1 LLM engine (v0.8.5) with config: model='Qwen/Qwen2.5-1.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=2560

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.19it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.19it/s]



INFO 11-27 21:20:53 [loader.py:458] Loading weights took 0.89 seconds
INFO 11-27 21:20:53 [gpu_model_runner.py:1347] Model loading took 2.8876 GiB and 3.648999 seconds
INFO 11-27 21:20:54 [kv_cache_utils.py:634] GPU KV cache size: 1,442,832 tokens
INFO 11-27 21:20:54 [kv_cache_utils.py:637] Maximum concurrency for 2,560 tokens per request: 563.61x
INFO 11-27 21:20:54 [core.py:159] init engine (profile, create kv cache, warmup model) took 1.13 seconds
INFO 11-27 21:20:54 [core_client.py:439] Core engine process 0 ready.
SimpleAgent инициализирован!

LoRA загружен: False
LoRA имя: None


In [6]:
# ============================================================================
# ДЕМОНСТРАЦИЯ ВЗАИМОДЕЙСТВИЯ АГЕНТА И ОРАКУЛА
# ============================================================================

print("=" * 80)
print("ДЕМОНСТРАЦИЯ ВЗАИМОДЕЙСТВИЯ АГЕНТА И ОРАКУЛА")
print("=" * 80)

# Reset среды
print("\n[1] Reset среды с ораклом...")
oracle_obs, oracle_info = oracle_val_envs.reset({})
print(f"   Instruction: {oracle_info[0].get('instruction', 'N/A')}")
print(f"   Observation keys: {list(oracle_obs.keys())}")

# Несколько шагов взаимодействия
num_steps = 5
print(f"\n[2] Выполняем {num_steps} шагов взаимодействия...\n")

for step_num in range(1, num_steps + 1):
    print(f"\n{'='*80}")
    print(f"ШАГ {step_num}/{num_steps}")
    print(f"{'='*80}")
    
    # Агент получает наблюдение
    current_obs = oracle_obs['text'][0]
    print(f"\n[Шаг {step_num}] Текущее наблюдение (первые 300 символов):")
    print(current_obs[:300] + "...")
    
    # Агент предсказывает действие (может включать вопрос к оракулу)
    print(f"\n[Шаг {step_num}] Агент предсказывает действие...")
    action_text = agent.predict(current_obs)
    print(f"   Действие агента: {action_text[:200]}...")
    
    # Выполняем step
    print(f"\n[Шаг {step_num}] Выполняем step...")
    oracle_obs, rewards, dones, infos = oracle_val_envs.step([action_text])
    
    # Показываем результаты
    print(f"   Reward: {rewards[0]:.4f}")
    print(f"   Done: {dones[0]}")
    
    # Проверяем, был ли задан вопрос оракулу
    if 'question' in infos[0] and infos[0]['question']:
        print(f"\n   [ОРАКУЛ] Задан вопрос: {infos[0]['question']}")
        if 'oracle_answer' in infos[0]:
            print(f"   [ОРАКУЛ] Ответ: {infos[0]['oracle_answer']}")
        else:
            print(f"   [ОРАКУЛ] Ответ не получен")
    else:
        print(f"\n   [ОРАКУЛ] Вопрос не задавался")
    
    # Проверяем, завершена ли задача
    if dones[0]:
        print(f"\n   ✓ Задача завершена!")
        if infos[0].get('won', False):
            print(f"   ✓ Задача выполнена успешно!")
        break
    
    print(f"\n   Следующее наблюдение будет включать ответ оракула (если был задан вопрос)")

print(f"\n{'='*80}")
print("ДЕМОНСТРАЦИЯ ЗАВЕРШЕНА")
print(f"{'='*80}")


ДЕМОНСТРАЦИЯ ВЗАИМОДЕЙСТВИЯ АГЕНТА И ОРАКУЛА

[1] Reset среды с ораклом...
(CraftextWorkerOracle pid=4032174) [DEBUG] Calling _jitted_reset. instruction_idx=0, reset_counter=1


(raylet) [2025-11-27 21:25:40,692 E 4032094 4032123] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2025-11-27_21-21-28_326942_4029626 is over 95% full, available space: 45.7451 GB; capacity: 1511.58 GB. Object creation will fail if spilling is required.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   Instruction: Collect wood.
   Observation keys: ['text', 'anchor']

[2] Выполняем 5 шагов взаимодействия...


ШАГ 1/5

[Шаг 1] Текущее наблюдение (первые 300 символов):

Your goal is to complete the following task:
**TASK:** Collect wood.

This is what you currently see:
Map: 
grass -4, 3 grass -3, 3 grass -2, 3 grass -1, 3 grass 0, 3 grass 1, 3 sand 2, 3 sand 3, 3 water 4, 3
grass -4, 2 grass -3, 2 grass -2, 2 grass -1, 2 grass 0, 2 grass 1, 2 grass 2, 2 sand 3, 2...

[Шаг 1] Агент предсказывает действие...


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.39it/s, est. speed input: 842.35 toks/s, output: 34.69 toks/s]


   Действие агента: <question>What is my current position and what can I do next?</question>
<action>noop</action>...

[Шаг 1] Выполняем step...
(CraftextWorkerOracle pid=4032174) [DEBUG] Calling _jitted_step. action=0, question=What is my current position and what can I do next?, state_shapes=TextEnvState(env_state=EnvState(map=(64, 64), mob_map=(64, 64), player_position=(2,), player_direction=(), player_health=(), player_food=(), player_drink=(), player_energy=(), is_sleeping=(), player_recover=(), player_hunger=(), player_thirst=(), player_fatigue=(), inventory=Inventory(wood=(), stone=(), coal=(), iron=(), diamond=(), sapling=(), wood_pickaxe=(), stone_pickaxe=(), iron_pickaxe=(), wood_sword=(), stone_sword=(), iron_sword=()), zombies=Mobs(position=(3, 2), health=(3,), mask=(3,), attack_cooldown=(3,)), cows=Mobs(position=(3, 2), health=(3,), mask=(3,), attack_cooldown=(3,)), skeletons=Mobs(position=(2, 2), health=(2,), mask=(2,), attack_cooldown=(2,)), arrows=Mobs(position=(3, 2), 

...

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  3.34it/s, est. speed input: 2606.93 toks/s, output: 70.27 toks/s]


   Действие агента: <question>What should I do next?</question>
<action>Collect more wood</action>...

[Шаг 5] Выполняем step...
(CraftextWorkerOracle pid=4032174) [DEBUG] Calling _jitted_step. action=-1, question=What should I do next?, state_shapes=TextEnvState(env_state=EnvState(map=(64, 64), mob_map=(64, 64), player_position=(2,), player_direction=(), player_health=(), player_food=(), player_drink=(), player_energy=(), is_sleeping=(), player_recover=(), player_hunger=(), player_thirst=(), player_fatigue=(), inventory=Inventory(wood=(), stone=(), coal=(), iron=(), diamond=(), sapling=(), wood_pickaxe=(), stone_pickaxe=(), iron_pickaxe=(), wood_sword=(), stone_sword=(), iron_sword=()), zombies=Mobs(position=(3, 2), health=(3,), mask=(3,), attack_cooldown=(3,)), cows=Mobs(position=(3, 2), health=(3,), mask=(3,), attack_cooldown=(3,)), skeletons=Mobs(position=(2, 2), health=(2,), mask=(2,), attack_cooldown=(2,)), arrows=Mobs(position=(3, 2), health=(3,), mask=(3,), attack_cooldown=(3,)

(raylet) [2025-11-27 21:26:30,715 E 4032094 4032123] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2025-11-27_21-21-28_326942_4029626 is over 95% full, available space: 45.745 GB; capacity: 1511.58 GB. Object creation will fail if spilling is required.


   Reward: 0.0000
   Done: False

   [ОРАКУЛ] Задан вопрос: What should I do next?
   [ОРАКУЛ] Ответ: Go towards the nearest resource and start mining if it's stone or iron; gather wood if it's diamond; or find a crafting table first before starting any other actions.
Based on the current map and resource availability, you should go towards the nearest resource and start mining if it's stone or iron; gather wood if it's diamond; or find a crafting table first before starting any other actions. To get started, head towards the nearest resource. If there's no diamond near you, focus on gathering wood since you can mine wood from trees. Once you've gathered enough wood, check if you can craft a pickaxe using a table. If not, move closer to potential sources of metal like diamonds or iron. In summary, prioritize finding a crafting table, then proceed with mining or gathering based on what's available within reach. Start by heading to the closest resource point. If you need to mine, choose 

(raylet) [2025-11-27 21:26:40,719 E 4032094 4032123] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2025-11-27_21-21-28_326942_4029626 is over 95% full, available space: 45.7449 GB; capacity: 1511.58 GB. Object creation will fail if spilling is required.
(raylet) [2025-11-27 21:26:50,724 E 4032094 4032123] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2025-11-27_21-21-28_326942_4029626 is over 95% full, available space: 45.7448 GB; capacity: 1511.58 GB. Object creation will fail if spilling is required.
(raylet) [2025-11-27 21:27:00,728 E 4032094 4032123] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2025-11-27_21-21-28_326942_4029626 is over 95% full, available space: 45.7448 GB; capacity: 1511.58 GB. Object creation will fail if spilling is required.
(raylet) [2025-11-27 21:27:10,733 E 4032094 4032123] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2025-11-27_21-21-28_326942_4029626 is over 95% full, available space: 45.7448 GB; capacity: 1511.58 GB. Obj

In [ ]:
# ============================================================================
# ДЕМОНСТРАЦИЯ ВЗАИМОДЕЙСТВИЯ АГЕНТА И ОРАКУЛА
# ============================================================================

print("=" * 80)
print("ДЕМОНСТРАЦИЯ ВЗАИМОДЕЙСТВИЯ АГЕНТА И ОРАКУЛА")
print("=" * 80)

# Reset среды
print("\n[1] Reset среды с ораклом...")
oracle_obs, oracle_info = oracle_val_envs.reset({})
print(f"   Instruction: {oracle_info[0].get('instruction', 'N/A')}")
print(f"   Observation keys: {list(oracle_obs.keys())}")

# Несколько шагов взаимодействия
num_steps = 5
print(f"\n[2] Выполняем {num_steps} шагов взаимодействия...\n")

for step_num in range(1, num_steps + 1):
    print(f"\n{'='*80}")
    print(f"ШАГ {step_num}/{num_steps}")
    print(f"{'='*80}")
    
    # Агент получает наблюдение
    current_obs = oracle_obs['text'][0]
    print(f"\n[Шаг {step_num}] Текущее наблюдение (первые 300 символов):")
    print(current_obs[:] + "...")
    
    # Агент предсказывает действие (может включать вопрос к оракулу)
    print(f"\n[Шаг {step_num}] Агент предсказывает действие...")
    action_text = agent.predict(current_obs)
    print(f"   Действие агента: {action_text[:200]}...")
    
    # Выполняем step
    print(f"\n[Шаг {step_num}] Выполняем step...")
    oracle_obs, rewards, dones, infos = oracle_val_envs.step([action_text])
    
    # Показываем результаты
    print(f"   Reward: {rewards[0]:.4f}")
    print(f"   Done: {dones[0]}")
    
    # Проверяем, был ли задан вопрос оракулу
    if 'question' in infos[0] and infos[0]['question']:
        print(f"\n   [ОРАКУЛ] Задан вопрос: {infos[0]['question']}")
        if 'oracle_answer' in infos[0]:
            print(f"   [ОРАКУЛ] Ответ: {infos[0]['oracle_answer']}")
        else:
            print(f"   [ОРАКУЛ] Ответ не получен")
    else:
        print(f"\n   [ОРАКУЛ] Вопрос не задавался")
    
    # Проверяем, завершена ли задача
    if dones[0]:
        print(f"\n   ✓ Задача завершена!")
        if infos[0].get('won', False):
            print(f"   ✓ Задача выполнена успешно!")
        break
    
    print(f"\n   Следующее наблюдение будет включать ответ оракула (если был задан вопрос)")

print(f"\n{'='*80}")
print("ДЕМОНСТРАЦИЯ ЗАВЕРШЕНА")
print(f"{'='*80}")


ДЕМОНСТРАЦИЯ ВЗАИМОДЕЙСТВИЯ АГЕНТА И ОРАКУЛА

[1] Reset среды с ораклом...
(CraftextWorkerOracle pid=4032177) [DEBUG] Calling _jitted_reset. instruction_idx=0, reset_counter=1
   Instruction: Collect wood.
   Observation keys: ['text', 'anchor']

[2] Выполняем 5 шагов взаимодействия...


ШАГ 1/5

[Шаг 1] Текущее наблюдение (первые 300 символов):

Your goal is to complete the following task:
**TASK:** Collect wood.

This is what you currently see:
Map: 
grass -4, 3 grass -3, 3 grass -2, 3 grass -1, 3 grass 0, 3 grass 1, 3 sand 2, 3 sand 3, 3 water 4, 3
grass -4, 2 grass -3, 2 grass -2, 2 grass -1, 2 grass 0, 2 grass 1, 2 grass 2, 2 sand 3, 2 sand 4, 2
grass -4, 1 grass -3, 1 grass -2, 1 grass -1, 1 grass 0, 1 grass 1, 1 grass 2, 1 grass 3, 1 sand 4, 1
grass -4, 0 grass -3, 0 grass -2, 0 grass -1, 0 agent 0, 0 grass 1, 0 grass 2, 0 grass 3, 0 grass 4, 0
grass -4, -1 grass -3, -1 grass -2, -1 grass -1, -1 grass 0, -1 grass 1, -1 grass 2, -1 grass 3, -1 sand 4, -1
grass -4, -2 grass -3, -2

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  3.21it/s, est. speed input: 1953.15 toks/s, output: 61.13 toks/s]


   Действие агента: <question>What is the next step in collecting wood?</question>
<action>noop...

[Шаг 1] Выполняем step...
(CraftextWorkerOracle pid=4032177) [DEBUG] Calling _jitted_step. action=-1, question=What is the next step in collecting wood?, state_shapes=TextEnvState(env_state=EnvState(map=(64, 64), mob_map=(64, 64), player_position=(2,), player_direction=(), player_health=(), player_food=(), player_drink=(), player_energy=(), is_sleeping=(), player_recover=(), player_hunger=(), player_thirst=(), player_fatigue=(), inventory=Inventory(wood=(), stone=(), coal=(), iron=(), diamond=(), sapling=(), wood_pickaxe=(), stone_pickaxe=(), iron_pickaxe=(), wood_sword=(), stone_sword=(), iron_sword=()), zombies=Mobs(position=(3, 2), health=(3,), mask=(3,), attack_cooldown=(3,)), cows=Mobs(position=(3, 2), health=(3,), mask=(3,), attack_cooldown=(3,)), skeletons=Mobs(position=(2, 2), health=(2,), mask=(2,), attack_cooldown=(2,)), arrows=Mobs(position=(3, 2), health=(3,), mask=(3,), atta

(raylet) [2025-11-27 21:44:41,168 E 4032094 4032123] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2025-11-27_21-21-28_326942_4029626 is over 95% full, available space: 45.7418 GB; capacity: 1511.58 GB. Object creation will fail if spilling is required.
(raylet) [2025-11-27 21:44:51,173 E 4032094 4032123] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2025-11-27_21-21-28_326942_4029626 is over 95% full, available space: 45.7418 GB; capacity: 1511.58 GB. Object creation will fail if spilling is required.


   Reward: 0.0000
   Done: False

   [ОРАКУЛ] Задан вопрос: What is the next step in collecting wood?
   [ОРАКУЛ] Ответ: To collect wood, stand next to a tree and use the DO action. I don't have information about other potential actions related to mining or crafting. To continue, you need to interact with the world by standing near a tree. Once there, simply use the DO action. Based solely on the information given, no specific tool is needed initially, just being next to a tree will allow you to start gathering wood. I don't have sufficient details to suggest any immediate progression beyond this basic interaction with the environment. Therefore, my response remains focused on starting the process of obtaining wood.

   Следующее наблюдение будет включать ответ оракула (если был задан вопрос)

ШАГ 2/5

[Шаг 2] Текущее наблюдение (первые 300 символов):

Your goal is to complete the following task:
**TASK:** Collect wood.

This is what you currently see:
Map: 
grass -4, 3 grass -3, 3 gra

...

(raylet) [2025-11-27 21:45:21,187 E 4032094 4032123] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2025-11-27_21-21-28_326942_4029626 is over 95% full, available space: 45.7418 GB; capacity: 1511.58 GB. Object creation will fail if spilling is required.


   Reward: 0.0000
   Done: False

   [ОРАКУЛ] Задан вопрос: What should I do next?
   [ОРАКУЛ] Ответ: Go to the nearest tree and use the DO action to collect wood. Based on the available options, it seems like you're missing a crafting table on the map which would allow you to craft tools. Since you only need wood to start mining stone, you can go to the nearest tree, collect some wood, then find a crafting table and craft a basic pickaxe. After obtaining the pickaxe, you will be able to mine stone by standing next to the stone and using the DO action again. If you don't see a crafting table immediately, you might want to search for one near the area where you plan to mine. Once you have a pickaxe and stone, you can move closer to the iron ore and try to mine it as well if possible. Remember to keep moving towards more resources as they get further away. Good luck!

   Следующее наблюдение будет включать ответ оракула (если был задан вопрос)

ШАГ 5/5

[Шаг 5] Текущее наблюдение (первые

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  3.56it/s, est. speed input: 2799.89 toks/s, output: 67.85 toks/s]


   Действие агента: <question>What should I do next?</question>
<action>right</action>...

[Шаг 5] Выполняем step...
(CraftextWorkerOracle pid=4032177) [DEBUG] Calling _jitted_step. action=2, question=What should I do next?, state_shapes=TextEnvState(env_state=EnvState(map=(64, 64), mob_map=(64, 64), player_position=(2,), player_direction=(), player_health=(), player_food=(), player_drink=(), player_energy=(), is_sleeping=(), player_recover=(), player_hunger=(), player_thirst=(), player_fatigue=(), inventory=Inventory(wood=(), stone=(), coal=(), iron=(), diamond=(), sapling=(), wood_pickaxe=(), stone_pickaxe=(), iron_pickaxe=(), wood_sword=(), stone_sword=(), iron_sword=()), zombies=Mobs(position=(3, 2), health=(3,), mask=(3,), attack_cooldown=(3,)), cows=Mobs(position=(3, 2), health=(3,), mask=(3,), attack_cooldown=(3,)), skeletons=Mobs(position=(2, 2), health=(2,), mask=(2,), attack_cooldown=(2,)), arrows=Mobs(position=(3, 2), health=(3,), mask=(3,), attack_cooldown=(3,)), arrow_dire

(raylet) [2025-11-27 21:45:31,191 E 4032094 4032123] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2025-11-27_21-21-28_326942_4029626 is over 95% full, available space: 45.7418 GB; capacity: 1511.58 GB. Object creation will fail if spilling is required.
(raylet) [2025-11-27 21:45:41,195 E 4032094 4032123] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2025-11-27_21-21-28_326942_4029626 is over 95% full, available space: 45.732 GB; capacity: 1511.58 GB. Object creation will fail if spilling is required.
(raylet) [2025-11-27 21:45:51,199 E 4032094 4032123] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2025-11-27_21-21-28_326942_4029626 is over 95% full, available space: 45.7319 GB; capacity: 1511.58 GB. Object creation will fail if spilling is required.
(raylet) [2025-11-27 21:46:01,203 E 4032094 4032123] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2025-11-27_21-21-28_326942_4029626 is over 95% full, available space: 45.7319 GB; capacity: 1511.58 GB. Obje

In [ ]:
# Пример step с вопросом к оракулу
print("=== Step с вопросом к оракулу ===")

# Формируем действие с вопросом (формат, который ожидает LLM)
text_action_with_question = """<question>What is the nearest diamond to me?</question>
<action>do</action>"""

# Выполняем step
next_obs, rewards, dones, infos = oracle_val_envs.step([text_action_with_question])

print(f"\nReward: {rewards[0]}")
print(f"Done: {dones[0]}")
print(f"\nInfo keys: {infos[0].keys()}")

# Проверяем, есть ли ответ оракла
if 'oracle_answer' in infos[0]:
    print(f"\n=== Ответ оракла ===")
    print(infos[0]['oracle_answer'])
else:
    print("\nОтвет оракла не найден в info")

# Проверяем, был ли задан вопрос
if 'question' in infos[0]:
    print(f"\n=== Заданный вопрос ===")
    print(infos[0]['question'])

# Показываем новое наблюдение (включает ответ оракла, если был)
print(f"\n=== Новое наблюдение (первые 800 символов) ===")
print(next_obs['text'][0])
print("\n...")

In [ ]:
# Пример step БЕЗ вопроса (обычное действие)
print("=== Step без вопроса ===")

text_action_no_question = "<action>right</action>"

next_obs2, rewards2, dones2, infos2 = oracle_val_envs.step([text_action_no_question])

print(f"Reward: {rewards2[0]}")
print(f"Done: {dones2[0]}")

# Проверяем, есть ли ответ оракла (не должно быть, т.к. вопрос не задавали)
if 'oracle_answer' in infos2[0]:
    print(f"\nОтвет оракла: {infos2[0]['oracle_answer']}")
else:
    print("\nОтвет оракла отсутствует (вопрос не задавался)")

print(f"\n=== Новое наблюдение (первые 500 символов) ===")
print(next_obs2['text'][0])


In [ ]:
# Пример нескольких шагов с разными вопросами
print("=== Несколько шагов с вопросами ===\n")

# Шаг 1: Вопрос о том, что делать дальше
action1 = """<question>What should I do next to complete my task?</question>
<action>up</action>"""
obs1, rew1, done1, info1 = oracle_val_envs.step([action1])
print(f"Шаг 1 - Вопрос: {info1[0].get('question', 'N/A')}")
print(f"Ответ оракла: {info1[0].get('oracle_answer', 'N/A')[:200]}...")
print(f"Reward: {rew1[0]}\n")

# Шаг 2: Вопрос о ресурсах
action2 = """<question>Where can I find wood nearby?</question>
<action>left</action>"""
obs2, rew2, done2, info2 = oracle_val_envs.step([action2])
print(f"Шаг 2 - Вопрос: {info2[0].get('question', 'N/A')}")
print(f"Ответ оракла: {info2[0].get('oracle_answer', 'N/A')[:200]}...")
print(f"Reward: {rew2[0]}\n")

# Шаг 3: Действие без вопроса
action3 = "<action>do</action>"
obs3, rew3, done3, info3 = oracle_val_envs.step([action3])
print(f"Шаг 3 - Без вопроса")
print(f"Ответ оракла: {info3[0].get('oracle_answer', 'Отсутствует (вопрос не задавался)')}")
print(f"Reward: {rew3[0]}\n")


In [ ]:
# Сравнение структуры наблюдений: базовая vs oracle версия
print("=== Сравнение структур ===")

# Базовая версия
base_obs, base_info = val_envs.reset({})
print("\nБазовая версия:")
print(f"  Observation keys: {list(base_obs.keys())}")
print(f"  Info keys: {list(base_info[0].keys())}")
print(f"  Есть oracle_answer? {('oracle_answer' in base_info[0])}")

# Oracle версия
oracle_obs, oracle_info = oracle_val_envs.reset({})
print("\nOracle версия:")
print(f"  Observation keys: {list(oracle_obs.keys())}")
print(f"  Info keys: {list(oracle_info[0].keys())}")
print(f"  Есть oracle_answer? {('oracle_answer' in oracle_info[0])}")

print("\n=== Разница ===")
print("Oracle версия добавляет:")
print("  - 'oracle_answer' в info (если был задан вопрос)")
print("  - 'question' в info (если был задан вопрос)")
print("  - Ответ оракла включается в text observation в следующем шаге")


In [ ]:
obs, info = val_envs.reset({})
state = ray.get(val_envs.envs._workers[0].get_state.remote())

In [ ]:
# Импортируем функции для работы с картой
from craftext_utility import get_map_info, find_nearest_block
from craftax.craftax_classic.constants import BlockType
import jax

In [ ]:

# Получаем env_state из state (если state - это TextEnvState)
if hasattr(state, 'env_state'):
    env_state = jax.device_get(state.env_state)
else:
    env_state = jax.device_get(state)

# Получаем информацию о всей карте
map_info = get_map_info(env_state)
print(f"Размер карты: {map_info['shape']}")
print(f"Позиция игрока: {map_info['player_position']}")

# Ищем все алмазы на карте
diamond_info = get_map_info(env_state, BlockType.DIAMOND)
print(f"\nНайдено алмазов: {diamond_info['block_count']}")
if diamond_info['block_count'] > 0:
    print(f"Позиции всех алмазов: {diamond_info['block_positions']}")
    print(f"\nБлижайший алмаз:")
    print(f"  Позиция: {diamond_info['nearest_block']['position']}")
    print(f"  Расстояние: {diamond_info['nearest_block']['distance']} шагов")
    print(f"  Относительная позиция: {diamond_info['nearest_block']['relative_position']}")

# Используем удобную функцию для поиска ближайшего алмаза
nearest_diamond = find_nearest_block(env_state, BlockType.DIAMOND)
if nearest_diamond:
    print(f"\nБлижайший алмаз находится {nearest_diamond['direction']} на расстоянии {nearest_diamond['distance']} шагов")
    print(f"Абсолютная позиция: {nearest_diamond['position']}")
else:
    print("\nАлмазы на карте не найдены")


In [ ]:
print("=== Поиск различных ресурсов на карте ===\n")

# Список интересных блоков для поиска
interesting_blocks = [
    (BlockType.DIAMOND, "Алмаз"),
    (BlockType.IRON, "Железо"),
    (BlockType.COAL, "Уголь"),
    (BlockType.STONE, "Камень"),
    (BlockType.TREE, "Дерево"),
    (BlockType.CRAFTING_TABLE, "Верстак"),
    (BlockType.FURNACE, "Печь"),
]

for block_type, name in interesting_blocks:
    nearest = find_nearest_block(env_state, block_type)
    if nearest:
        print(f"{name}: {nearest['direction']}, расстояние {nearest['distance']} шагов, позиция {nearest['position']}")
    else:
        print(f"{name}: не найдено на карте")

# Можно также получить полную информацию о карте для любого блока
print("\n=== Полная информация о карте ===")
full_map_info = get_map_info(env_state)
print(f"Размер карты: {full_map_info['shape']}")
print(f"Позиция игрока: {full_map_info['player_position']}")
print(f"Карта доступна в переменной map_info['map']")


In [ ]:
oracle_game_knwoledge_prompt = """
# Role
You are an expert guide for the CrafText (based on Craftax) environment. Your goal is to answer questions short and concise.

# Game Mechanics & Rules

### Progression & Mining Requirements:
- **To collect wood:** No tool required.
- **To mine stone:** Requires a `wood_pickaxe` in inventory.
- **To mine iron:** Requires a `stone_pickaxe` in inventory.
- **To mine diamond:** Requires an `iron_pickaxe` in inventory.

### Crafting & Placing Requirements:
- **To craft wood items (e.g., wood_pickaxe):** Requires `1 wood` and a `table`. You must be adjacent to the `table`.
- **To craft stone items (e.g., stone_pickaxe):** Requires `1 wood`, `1 stone` and a `table`. You must be adjacent to the `table`.
- **To craft iron items (e.g., iron_pickaxe):** Requires `1 wood`, `1 stone`, `1 coal`, `1 iron`, and a `table` and `furnace`. You must be adjacent to both.
- **To place table:** Requires `2 wood` in inventory.
- **To place furnace:** Requires `1 stone` in inventory.
- **To gather sapling:** You must stand on a `grass` block and use `DO`.

"""